# 외부 데이터 수집과 품질 점검


## 학습 목표

<!-- exercise-guidance:microprocessor/web-scraping/objective -->
페이지의 일부 필드가 빠져도 다른 기록까지 버리지 않도록, HTML 구조와 추출 규칙을 함께 확인합니다.
<!-- /exercise-guidance:microprocessor/web-scraping/objective -->


## JSON에서 필요한 값 고르기

<!-- exercise-guidance:microprocessor/web-scraping/activity-1 -->
### 시작 조건

센서 응답은 device·timestamp·temperature_c를 포함합니다. 두 번째 행은 온도가 빠지고 세 번째는 문자열 숫자입니다.

### 진행 방법

1. 제공된 JSON을 읽고 각 행의 필드·자료형을 확인하세요.
2. 수집에 성공한 것과 분석할 수 있는 값인 것을 구분하세요.

### 남길 결과

- 정상·누락·형식 변환 사례 표.
<!-- /exercise-guidance:microprocessor/web-scraping/activity-1 -->


In [ ]:
import json
payload='[{"device":"a","timestamp":0,"temperature_c":21.2},{"device":"a","timestamp":1},{"device":"b","timestamp":0,"temperature_c":"22.1"}]'
rows=json.loads(payload)
def clean(row):
    if 'temperature_c' not in row:return None,'missing temperature'
    try:return dict(row,temperature_c=float(row['temperature_c'])),'ok'
    except (ValueError,TypeError):return None,'invalid temperature'
for row in rows:print(clean(row))


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/web-scraping/activity-1-record -->
**행:**

**원래 값·자료형:**

**정리한 값:**

**처리 이유:**

### 확인할 점

- 누락된 온도를 실제 0도로 처리하지 않았는가?
- 변환 실패를 조용히 버리지 않고 이유를 남겼는가?
<!-- /exercise-guidance:microprocessor/web-scraping/activity-1-record -->


## 응답의 구조가 바뀌면

<!-- exercise-guidance:microprocessor/web-scraping/activity-2 -->
### 시작 조건

추가 입력 `temperature_c=null`, "error", 1000과 새 필드 unit="F"를 시험합니다.

### 진행 방법

1. 같은 `variants`의 네 행을 처리하세요. 먼저 숫자 변환 결과를 확인하고, 변환 실패는 원래 값과 오류 이유를 함께 기록하세요.
2. 섭씨로 해석할 수 있는 단위와 센서의 유효 온도 범위를 검증 함수의 인자로 정하세요. 선택한 단위·범위·경계 포함 여부를 적고, 범위 밖 값과 단위가 다른 행을 구분하세요. 센서 사양이 주어지지 않았으므로 이 범위는 본인이 정한 검증 조건으로 표시하세요.
3. 입력값·단위·변환한 값·통과 여부·거부 이유를 행별로 적으세요. 정상 행과 거부 행을 따로 모으고, 센서 고장 여부를 판단하려면 추가로 필요한 정보를 적으세요.

### 남길 결과

- 검증 함수와 오류 사례 표.
<!-- /exercise-guidance:microprocessor/web-scraping/activity-2 -->


In [ ]:
variants=[dict(device='a',timestamp=2,temperature_c=None),dict(device='a',timestamp=3,temperature_c='error'),dict(device='a',timestamp=4,temperature_c=1000),dict(device='a',timestamp=5,temperature_c=70,unit='F')]
# TODO: clean을 검증 단계별로 확장하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/web-scraping/activity-2-record -->
**입력:**

**형식 검사:**

**범위·단위 검사:**

**처리:**

### 확인할 점

- 형식이 숫자라는 이유만으로 센서 범위 검사를 생략하지 않았는가?
- 의미를 모르는 단위를 임의로 변환하지 않았는가?
<!-- /exercise-guidance:microprocessor/web-scraping/activity-2-record -->


## HTML만 제공되는 자료와 비교하기

<!-- exercise-guidance:microprocessor/web-scraping/activity-3 -->
### 시작 조건

API 응답이 없고 HTML 표만 있는 상황을 비교합니다. 기존 Selector 예제는 참고 자료로 둡니다.

### 진행 방법

1. API 필드와 HTML Selector가 깨지는 원인을 각각 한 가지 적으세요.
2. 수집·정리·저장을 함수 세 개로 나누어 입력과 출력을 정하세요.
3. 정리한 온도 자료를 CSV로 저장하고 다시 읽어 값·누락 처리가 유지되는지 보세요.

### 남길 결과

- 수집 파이프라인 그림과 CSV 왕복 확인.
<!-- /exercise-guidance:microprocessor/web-scraping/activity-3 -->


In [ ]:
import csv,io
valid=[clean(row)[0] for row in rows if clean(row)[0] is not None]
stream=io.StringIO();writer=csv.DictWriter(stream,fieldnames=['device','timestamp','temperature_c'])
writer.writeheader();writer.writerows(valid);stream.seek(0)
print(list(csv.DictReader(stream)))


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/web-scraping/activity-3-record -->
**단계:**

**입력:**

**출력:**

**확인할 오류:**

### 확인할 점

- 외부 사이트를 반복 요청하지 않고 고정 응답으로 처리 규칙을 시험했는가?
- 자료 이용 조건과 수집 간격은 실제 연결 전에 확인하도록 구분했는가?
<!-- /exercise-guidance:microprocessor/web-scraping/activity-3-record -->
